In [ ]:
%%capture

!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-gpu
!pip install -q pypdf

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

path = "/content/drive/MyDrive/rag_data"

files = os.listdir(path)

print(f"Total files found: {len(files)}")
print(files[:10])  # show first 10 files

Total files found: 61
['etf.pdf', 'Smart Beta, Direct.pdf', 'mutual funds.pdf', 'sec.pdf', 'jp.pdf', 'mutual f.pdf', 'etff.pdf', 'x.pdf', 'etfff.pdf', 'secc.pdf']


In [ ]:
!pip install -q pymupdf

In [ ]:
!apt-get update -qq
!apt-get install -qq tesseract-ocr poppler-utils

!pip install -q pytesseract pdf2image pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up libpoppler118:amd64 (22.02.0-2ubuntu0.13) ...
Setting up poppler-

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
import os

pdf_path = "/content/drive/MyDrive/rag_data"

documents = []

for file in os.listdir(pdf_path):
    if file.endswith(".pdf"):
        file_path = os.path.join(pdf_path, file)

        try:
            loader = PyMuPDFLoader(file_path)
            pages = loader.load()

            documents.extend(pages)

            print(f"Loaded {file}: {len(pages)} pages")

        except Exception as e:
            print(f"Error loading {file}: {e}")

print("\n-----------------------------")
print(f"Total pages loaded: {len(documents)}")

Loaded etf.pdf: 20 pages
Loaded Smart Beta, Direct.pdf: 51 pages
Loaded mutual funds.pdf: 50 pages
Loaded sec.pdf: 56 pages
Loaded jp.pdf: 8 pages
Loaded mutual f.pdf: 40 pages
Loaded etff.pdf: 27 pages
Loaded x.pdf: 8 pages
Loaded etfff.pdf: 60 pages
Loaded secc.pdf: 32 pages
Loaded sebi.pdf: 73 pages
Loaded 1978.pdf: 8 pages
Loaded 1977.pdf: 6 pages
Loaded 1979.pdf: 12 pages
Loaded 1980.pdf: 14 pages
Loaded 1981.pdf: 13 pages
Loaded 1982.pdf: 13 pages
Loaded 1983.pdf: 19 pages
Loaded 1984.pdf: 22 pages
Loaded 1985.pdf: 24 pages
Loaded 1986.pdf: 24 pages
Loaded 1987.pdf: 22 pages
Loaded 1988.pdf: 21 pages
Loaded 1989.pdf: 25 pages
Loaded 1990.pdf: 21 pages
Loaded 1991.pdf: 16 pages
Loaded 1992.pdf: 20 pages
Loaded 1993.pdf: 20 pages
Loaded 1994.pdf: 17 pages
Loaded 1995.pdf: 21 pages
Loaded 1996.pdf: 19 pages
Loaded 1997.pdf: 17 pages
Loaded 1998.pdf: 1 pages
Loaded 1999.pdf: 1 pages
Loaded 2000.pdf: 1 pages
Loaded 2001.pdf: 1 pages
Loaded 2002.pdf: 1 pages
Loaded 2003ltr.pdf: 22 page

In [ ]:
import os
import pytesseract
from pdf2image import convert_from_path
from langchain_core.documents import Document

pdf_path = "/content/drive/MyDrive/rag_data"

# PDFs that had 100% empty pages
bad_pdfs = [
    "knl.pdf",
    "sebi.pdf",
    "etfff.pdf",
    "sec.pdf",
    "Smart Beta, Direct.pdf",
    "mutual funds.pdf",
    "mutual f.pdf",
    "secc.pdf",
    "etff.pdf",
    "etf.pdf",
    "jp.pdf",
    "x.pdf"
]

ocr_documents = []

for pdf_file in bad_pdfs:
    file_path = os.path.join(pdf_path, pdf_file)

    print(f"\nProcessing {pdf_file}...")

    try:
        # Convert PDF pages into images
        pages = convert_from_path(
            file_path,
            dpi=200
        )

        print(f"Total pages: {len(pages)}")

        for page_num, image in enumerate(pages):

            # Extract text from image
            text = pytesseract.image_to_string(
                image,
                lang="eng"
            )

            text = text.strip()

            if text:
                ocr_documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": pdf_file,
                            "page": page_num,
                            "extraction": "OCR"
                        }
                    )
                )

        print(
            f"Extracted pages: {len([d for d in ocr_documents if d.metadata['source'] == pdf_file])}"
        )

    except Exception as e:
        print(f"Error processing {pdf_file}: {e}")


print("\n==============================")
print("OCR Complete")
print(f"Total OCR pages extracted: {len(ocr_documents)}")


Processing knl.pdf...
Total pages: 340
Extracted pages: 340

Processing sebi.pdf...
Total pages: 73
Extracted pages: 73

Processing etfff.pdf...
Total pages: 60
Extracted pages: 56

Processing sec.pdf...
Total pages: 56
Extracted pages: 54

Processing Smart Beta, Direct.pdf...
Total pages: 51
Extracted pages: 51

Processing mutual funds.pdf...
Total pages: 50
Extracted pages: 50

Processing mutual f.pdf...
Total pages: 40
Extracted pages: 40

Processing secc.pdf...
Total pages: 32
Extracted pages: 29

Processing etff.pdf...
Total pages: 27
Extracted pages: 27

Processing etf.pdf...
Total pages: 20
Extracted pages: 20

Processing jp.pdf...
Total pages: 8
Extracted pages: 8

Processing x.pdf...
Total pages: 8
Extracted pages: 8

OCR Complete
Total OCR pages extracted: 756


In [ ]:
import os
import pickle

# Folder to store processed data
save_path = "/content/drive/MyDrive/rag_processed"

os.makedirs(save_path, exist_ok=True)

# Save OCR extracted documents
ocr_file = os.path.join(save_path, "ocr_documents.pkl")

with open(ocr_file, "wb") as f:
    pickle.dump(ocr_documents, f)

print("✅ OCR documents saved successfully!")
print(f"Location: {ocr_file}")
print(f"Total OCR pages saved: {len(ocr_documents)}")

✅ OCR documents saved successfully!
Location: /content/drive/MyDrive/rag_processed/ocr_documents.pkl
Total OCR pages saved: 756


In [ ]:
import pickle

with open("/content/drive/MyDrive/rag_processed/raw_documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("✅ Raw documents saved")

✅ Raw documents saved


In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = "/content/drive/MyDrive/rag_data"

documents = []

for file in os.listdir(pdf_path):
    if file.endswith(".pdf"):
        file_path = os.path.join(pdf_path, file)

        try:
            loader = PyMuPDFLoader(file_path)
            pages = loader.load()
            documents.extend(pages)

        except Exception as e:
            print(f"Error loading {file}: {e}")

print("Loaded pages:", len(documents))

/tmp/ipykernel_8876/2899397857.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Loaded pages: 2388


In [ ]:
clean_documents = [
    doc for doc in documents
    if doc.page_content.strip()
]

print("Normal extracted pages:", len(clean_documents))
print("OCR recovered pages:", len(ocr_documents))

final_documents = clean_documents + ocr_documents

print("\n✅ Final corpus ready")
print("Total pages:", len(final_documents))

Normal extracted pages: 1623
OCR recovered pages: 756

✅ Final corpus ready
Total pages: 2379


In [ ]:
with open("/content/drive/MyDrive/rag_processed/final_documents.pkl", "wb") as f:
    pickle.dump(final_documents, f)

print("✅ Final corpus permanently saved")
print("Total documents:", len(final_documents))

NameError: name 'final_documents' is not defined

In [ ]:
from collections import defaultdict
import os

pdf_stats = defaultdict(lambda: {"total": 0, "empty": 0})

for doc in documents:
    source = os.path.basename(doc.metadata["source"])

    pdf_stats[source]["total"] += 1

    if len(doc.page_content.strip()) == 0:
        pdf_stats[source]["empty"] += 1

print("PDFs with empty pages:\n")

for pdf, stats in sorted(
    pdf_stats.items(),
    key=lambda x: x[1]["empty"],
    reverse=True
):
    if stats["empty"] > 0:
        percentage = stats["empty"] / stats["total"] * 100

        print(
            f"{pdf}: "
            f"{stats['empty']}/{stats['total']} empty "
            f"({percentage:.1f}%)"
        )

PDFs with empty pages:

knl.pdf: 340/340 empty (100.0%)
sebi.pdf: 73/73 empty (100.0%)
etfff.pdf: 60/60 empty (100.0%)
sec.pdf: 56/56 empty (100.0%)
Smart Beta, Direct.pdf: 51/51 empty (100.0%)
mutual funds.pdf: 50/50 empty (100.0%)
mutual f.pdf: 40/40 empty (100.0%)
secc.pdf: 32/32 empty (100.0%)
etff.pdf: 27/27 empty (100.0%)
etf.pdf: 20/20 empty (100.0%)
jp.pdf: 8/8 empty (100.0%)
x.pdf: 8/8 empty (100.0%)


In [ ]:
import re

cleaned_documents = []

for doc in final_documents:
    text = doc.page_content

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    # Skip very small chunks (covers, blank pages, etc.)
    if len(text) < 100:
        continue

    doc.page_content = text
    cleaned_documents.append(doc)

print("Original documents:", len(final_documents))
print("Cleaned documents:", len(cleaned_documents))
print("Removed:", len(final_documents) - len(cleaned_documents))

NameError: name 'final_documents' is not defined

In [ ]:
!find /content/drive/MyDrive -name "*.pkl" -ls

  5641671      0 -rw-r--r--   1 root     root            0 Jun 20 19:40 /content/drive/MyDrive/rag_processed/cleaned_documents.pkl
  5641678      0 -rw-r--r--   1 root     root            0 Jun 20 19:40 /content/drive/MyDrive/rag_processed/final_documents.pkl


In [ ]:
!find /content/drive/MyDrive -name "*.pdf" | wc -l

0


In [ ]:
import os

file_path = "/content/drive/MyDrive/rag_processed/final_documents.pkl"

print("Exists:", os.path.exists(file_path))
print("Size:", os.path.getsize(file_path))
print("Size MB:", os.path.getsize(file_path)/(1024*1024))

Exists: True
Size: 0
Size MB: 0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -lah /content/drive/MyDrive

total 7.7M
-rw------- 1 root root 237K Apr 10 05:38  2023CSD0632_Bachelor_of_Technology_Computer_Science_and_Engineering__YearSession_2025-2026-DECEMBER-202526Odd_REGULAR_Term_5_Grade_card.pdf
-rw------- 1 root root 150K Nov 12  2025 'Aviation Bay RESEARCH.pdf'
-rw------- 1 root root  184 Dec 23  2024  Bci.gdoc
drwx------ 2 root root 4.0K Jul 20  2020  Classroom
drwx------ 2 root root 4.0K Jun 20 10:53 'Colab Notebooks'
-rw------- 1 root root  24K Jun 20 19:57  embeddings.ipynb
-rw------- 1 root root 888K Jul 15  2024 'IMG_20240408_020026 (1).jpg'
-rw------- 1 root root 888K Jul 15  2024  IMG_20240408_020026.jpg
-rw------- 1 root root 156K Feb 13  2025  IMG_20250213_200637.jpg
-rw------- 1 root root  184 May  6 15:09 'Internship Tracker.gdoc'
drwx------ 2 root root 4.0K Jun 20 17:31  rag_data
drwx------ 2 root root 4.0K Jun 20 19:31  rag_processed
-rw------- 1 root root 186K Sep 22  2025 'result 4 (1).pdf'
-rw------- 1 root root 186K Sep 22  2025 'result 4.pdf'
-rw------- 1 root root 2